In [1]:
import os
if not hasattr(os, "_cwd"):
    os.chdir("../")
    os._cwd = True

In [2]:
import os
import sys

sys.path.append(os.getcwd())


from PIL import Image
from io import BytesIO
from utilities.Storage import Storage

from Environment import *


s3 = Storage()



input_path = "assets/logo_gtr.png"
input_height = 200


MAX_SIZE = 2000


# check if height is valid
if input_height > MAX_SIZE or input_height <= 0:
    print("Invalid height")
    # return 0

# check if object exists
if not s3.object_exists(MINIO_BUCKET_PUBLIC, input_path):
    print("Object does not exist")
    # return 0


file = s3.get_object(MINIO_BUCKET_PUBLIC, input_path)
old_image = Image.open(BytesIO(file.read()))


output_width = int(old_image.width * input_height / old_image.height)
new_image = old_image.resize((output_width, input_height), Image.Resampling.LANCZOS)



# remove old image if exists
output_path = f"{input_height}/{input_path}"
if s3.object_exists("public", output_path):
    s3.remove_object("public", output_path)



# save new image to s3
image_buffer = BytesIO()
new_image.save(image_buffer, format="PNG")
image_buffer.seek(0)
s3.put_object(
    "public",
    output_path,
    image_buffer,
    image_buffer.getbuffer().nbytes,  # get size of byte array
)




ObjectWriteResult(bucket_name='public', object_name='200/assets/logo_gtr.png', version_id=None, etag='4a10dc29de90ade4c770df3b7d1b5779', http_headers=HTTPHeaderDict({'Accept-Ranges': 'bytes', 'Content-Length': '0', 'ETag': '"4a10dc29de90ade4c770df3b7d1b5779"', 'Server': 'MinIO', 'Strict-Transport-Security': 'max-age=31536000; includeSubDomains', 'Vary': 'Origin, Accept-Encoding', 'X-Amz-Id-2': 'dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8', 'X-Amz-Request-Id': '189BAF8CCD036984', 'X-Content-Type-Options': 'nosniff', 'X-Ratelimit-Limit': '4044', 'X-Ratelimit-Remaining': '4044', 'X-Xss-Protection': '1; mode=block', 'Date': 'Wed, 11 Mar 2026 04:38:17 GMT'}), last_modified=None, location=None)

In [8]:
import os
import sys

sys.path.append(os.getcwd())


from PIL import Image
from io import BytesIO
from utilities.Storage import Storage

from Environment import *


class Converter:
    
    s3 = Storage()
    MAX_SIZE = 2000

    def to_thumbnail(self, input_path: str, input_height: int):

        # check if height is valid
        if input_height > self.MAX_SIZE or input_height <= 0:
            print("Invalid height")
            return 0

        # check if object exists
        if not self.s3.object_exists(MINIO_PUBLIC, input_path):
            print("Object does not exist")
            return 0


        file = self.s3.get_object(MINIO_PUBLIC, input_path)
        old_image = Image.open(BytesIO(file.read()))


        output_width = int(old_image.width * input_height / old_image.height)
        new_image = old_image.resize((output_width, input_height), Image.Resampling.LANCZOS)



        # remove old image if exists
        output_path = f"{input_height}/{input_path}"
        if self.s3.object_exists(MINIO_PUBLIC, output_path):
            self.s3.remove_object(MINIO_PUBLIC, output_path)



        # save new image to s3
        image_buffer = BytesIO()
        new_image.save(image_buffer, format="PNG")
        image_buffer.seek(0)
        self.s3.put_object(
            MINIO_PUBLIC,
            output_path,
            image_buffer,
            image_buffer.getbuffer().nbytes,  
        )

convert = Converter()

# convert.to_thumbnail("assets/logo_itc.png", 100)
# convert.to_thumbnail("assets/logo_itc.png", 200)

In [ ]:
s3 = Storage()

# list all objects in public bucket and assets folder
objects = s3.list_objects(MINIO_PUBLIC, prefix="assets/")
for obj in objects:
    print(obj.object_name)
    convert.to_thumbnail(obj.object_name, 100)
    convert.to_thumbnail(obj.object_name, 200)

assets/1riel.png
assets/apple.png
assets/background.png
assets/banana.png
assets/cherry.png
